# Lab 06 – Task 2: Predict the age of abalone using kNN
Age = Rings + 1.5. Uses a local `abalone.data` if present, else downloads from UCI, else OpenML.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import os
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
cols = ['Sex','Length','Diameter','Height','Whole weight','Shucked weight','Viscera weight','Shell weight','Rings']
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data'
try:
    df = pd.read_csv('abalone.data' if os.path.exists('abalone.data') else url, header=None, names=cols)
except Exception:
    from sklearn.datasets import fetch_openml
    df = fetch_openml('abalone', version=1, as_frame=True).frame
    df.columns = cols
df['Rings'] = df['Rings'].astype(int)
df['Age'] = df['Rings'] + 1.5
df.head()

In [ ]:
print(df.isnull().sum().sum(), 'missing values')
plt.figure(figsize=(9,6)); sns.heatmap(df.drop(columns='Sex').astype(float).corr(), annot=True, cmap='coolwarm'); plt.show()

In [ ]:
X = pd.get_dummies(df.drop(columns=['Rings','Age']), columns=['Sex'])
y = df['Rings']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
sc = StandardScaler().fit(X_train); Xtr, Xte = sc.transform(X_train), sc.transform(X_test)

ks = range(1, 41); rmse = []; r2 = []
for k in ks:
    p = KNeighborsRegressor(n_neighbors=k).fit(Xtr, y_train).predict(Xte)
    rmse.append(np.sqrt(mean_squared_error(y_test, p))); r2.append(r2_score(y_test, p))
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(ks, rmse, marker='o'); ax[0].set_title('RMSE vs K')
ax[1].plot(ks, r2, marker='o', color='g'); ax[1].set_title('R² vs K'); plt.show()
bk = ks[int(np.argmin(rmse))]
print('Best K:', bk, '| RMSE:', min(rmse), '| R²:', r2[bk-1])

In [ ]:
final = KNeighborsRegressor(n_neighbors=bk, weights='distance').fit(Xtr, y_train)
pred = final.predict(Xte)
print('Final RMSE:', np.sqrt(mean_squared_error(y_test, pred)), '| R²:', r2_score(y_test, pred))
plt.scatter(y_test, pred, alpha=0.5); plt.plot([0,30],[0,30],'r--'); plt.xlabel('Actual rings'); plt.ylabel('Predicted rings'); plt.show()
print('Predicted ages (years):', (pred[:5] + 1.5).round(1))